### Imports

In [2]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    IMG_SIZE,
    BATCH_SIZE,
    create_dataset
)

PROCESSED_PATH = PROJECT_ROOT / "dataset" / "processed"

CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___healthy"
]

print("TensorFlow:", tf.__version__)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)

TensorFlow: 2.21.0
Image size: (224, 224)
Batch size: 16


### Load dataset metadata

In [3]:
df = pd.read_csv(PROCESSED_PATH / "image_metadata.csv")

print("Total images:", len(df))
df.head()

Total images: 7579


,image_path,class_name
0,d:\Coding Files\PlantDiseaseDetection\dataset\...,Tomato_Bacterial_spot
1,d:\Coding Files\PlantDiseaseDetection\dataset\...,Tomato_Bacterial_spot
2,d:\Coding Files\PlantDiseaseDetection\dataset\...,Tomato_Bacterial_spot
3,d:\Coding Files\PlantDiseaseDetection\dataset\...,Tomato_Bacterial_spot
4,d:\Coding Files\PlantDiseaseDetection\dataset\...,Tomato_Bacterial_spot


### Split the dataset
#### We'll use stratified splitting so that each class is represented proportionally in all three datasets.

In [4]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["class_name"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["class_name"]
)

print("Training images:", len(train_df))
print("Validation images:", len(validation_df))
print("Testing images:", len(test_df))

Training images: 5305
Validation images: 1137
Testing images: 1137


### Verify class distribution

In [5]:
print("Training distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation distribution:")
print(validation_df["class_name"].value_counts())

print("\nTesting distribution:")
print(test_df["class_name"].value_counts())

Training distribution:
class_name
Tomato_Bacterial_spot    1489
Tomato_Late_blight       1336
Tomato_healthy           1114
Tomato_Early_blight       700
Tomato_Leaf_Mold          666
Name: count, dtype: int64

Validation distribution:
class_name
Tomato_Bacterial_spot    319
Tomato_Late_blight       286
Tomato_healthy           239
Tomato_Early_blight      150
Tomato_Leaf_Mold         143
Name: count, dtype: int64

Testing distribution:
class_name
Tomato_Bacterial_spot    319
Tomato_Late_blight       287
Tomato_healthy           238
Tomato_Early_blight      150
Tomato_Leaf_Mold         143
Name: count, dtype: int64


### Save the splits

In [6]:
train_df.to_csv(PROCESSED_PATH / "train.csv", index=False)
validation_df.to_csv(PROCESSED_PATH / "validation.csv", index=False)
test_df.to_csv(PROCESSED_PATH / "test.csv", index=False)

print("All three dataset splits saved successfully.")

All three dataset splits saved successfully.


### Create TensorFlow datasets

In [7]:
train_dataset = create_dataset(
    train_df,
    CLASSES,
    training=True
)

validation_dataset = create_dataset(
    validation_df,
    CLASSES,
    training=False
)

test_dataset = create_dataset(
    test_df,
    CLASSES,
    training=False
)

print("TensorFlow datasets created successfully.")

TensorFlow datasets created successfully.


### Check one batch

In [8]:
for images, labels in train_dataset.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Minimum pixel value:", tf.reduce_min(images).numpy())
    print("Maximum pixel value:", tf.reduce_max(images).numpy())

Image batch shape: (16, 224, 224, 3)
Label batch shape: (16,)
Minimum pixel value: 0.0
Maximum pixel value: 0.96556604


### Visualize preprocessed images

In [ ]:
plt.figure(figsize=(12, 8))

for images, labels in train_dataset.take(1):
    for i in range(min(8, len(images))):
        plt.subplot(2, 4, i + 1)

        plt.imshow(images[i].numpy())
        class_index = labels[i].numpy()

        plt.title(CLASSES[class_index].replace("Tomato___", ""))
        plt.axis("off")

plt.tight_layout()
plt.show()